# Metapath M-P-M-P collaborative filtering (architecture C) — results

**Compute runs externally**; this notebook only loads results and plots them.

profile-CF (3b) borrowed from CHEMICAL neighbours (x). Here molecule neighbours are defined by
**co-binding (y)** — "molecules that share receptors with me" — the classic M-P-M-P metapath. Bars:

- **raw** — `[mol | prot]`.
- **meta** — + co-binding-neighbour CF read at the target protein (scalars).
- **meta_full** — + the whole co-binding-neighbour profile over all P proteins (fat).

**Cold-start collapse (the honest inductive result):** a held-out molecule has no train row → no
co-binding signature → its metapath features are zeroed → `meta ≈ raw`. Pure-y molecule
neighbourhoods can't reach a cold molecule (same wall as MF / architecture B). Transductive both live.

Both regimes, **5 repeats**. Inductive cold split **stratified**. Binary weights. Embeddings standardised.

### Run
```bash
MAX_PARALLEL=10 bash scripts/modeling/eval/run_metapath_cf.sh
```

In [ ]:
import sys, pathlib, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import t as student_t

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('run from inside the orbind repository')
    ROOT = ROOT.parent

RESULT_DIR = ROOT / 'results/graph/metapath_cf'
COMBINED = RESULT_DIR / 'tables/metapath_cf_runs.csv'
RUNS_DIR = RESULT_DIR / 'tables/runs'
FIGURES_DIR = ROOT / 'notebooks/figures'; FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ['AUROC', 'AUPRC', 'MCC', 'F1']
REGIMES = ['transductive', 'inductive_molecule']

if COMBINED.exists():
    RES = pd.read_csv(COMBINED)
    print('loaded', COMBINED.relative_to(ROOT))
elif RUNS_DIR.exists() and glob.glob(str(RUNS_DIR / '*.csv')):
    RES = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RUNS_DIR / '*.csv')))], ignore_index=True)
    print(f'loaded {RES.repeat.nunique()} repeat-files from', RUNS_DIR.relative_to(ROOT))
else:
    RES = pd.DataFrame()
    print('No results yet.')

def ci95_half(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return float(student_t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

if len(RES):
    print('repeats per regime:', RES.groupby('regime').repeat.nunique().to_dict())
    print('mean test prevalence:', RES.groupby('regime').test_prevalence.mean().round(3).to_dict())
    diag = RES.groupby('regime')[['cobind_mean_sim','zero_signature_mols']].mean().round(3)
    print('\nmetapath diagnostics (co-binding neighbour similarity; fraction of zero-signature molecules):')
    display(diag)
    display(RES.groupby(['regime', 'bar'])[METRICS].mean().round(3))

In [ ]:
# ---- bar-grid: regimes (rows) x metrics (cols), mean +/- 95% t-CI over repeats ----
ORDER = ['raw', 'meta', 'meta_full']
LABEL = {'raw':'raw','meta':'meta','meta_full':'meta\\nfull'}
BAR_C = {'raw':'#9a9a95','meta':'#3d9970','meta_full':'#1f7a54'}

if not len(RES):
    print('No results to plot yet.')
else:
    fig, axes = plt.subplots(len(REGIMES), len(METRICS), figsize=(15, 7), squeeze=False)
    for ri, regime in enumerate(REGIMES):
        d = RES[RES.regime == regime]
        bars = [b for b in ORDER if b in set(d.bar)]
        for mi, metric in enumerate(METRICS):
            ax = axes[ri, mi]
            means = [d[d.bar == b][metric].mean() for b in bars]
            errs = [ci95_half(d[d.bar == b][metric]) for b in bars]
            x = np.arange(len(bars))
            ax.bar(x, means, color=[BAR_C[b] for b in bars], edgecolor='white', linewidth=1.2,
                   yerr=np.nan_to_num(errs), error_kw=dict(ecolor='#33322f', capsize=3, lw=1), zorder=3)
            if means:
                ax.axhline(means[0], color=BAR_C[bars[0]], ls='--', lw=1, zorder=1)
            ax.set_ylim(0, 1); ax.grid(alpha=.15, axis='y')
            ax.set_xticks(x); ax.set_xticklabels([LABEL[b] for b in bars], fontsize=8)
            for s in ('top', 'right'): ax.spines[s].set_visible(False)
            if ri == 0: ax.set_title(metric, fontsize=11)
            for xi, v in zip(x, means):
                if np.isfinite(v):
                    ax.text(xi, v + .02, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        nrep = d.repeat.nunique()
        axes[ri, 0].set_ylabel(f'{regime}\n({nrep} repeats)', fontsize=10)
    fig.suptitle('Metapath M-P-M-P (co-binding neighbours, y) — mean +/- 95% t-CI (stratified inductive split)', fontsize=12, y=1.0)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'metapath_cf.png', dpi=150, bbox_inches='tight')
    plt.show()

## How to read

- The comparison is **meta / meta_full vs raw in transductive** — does a *co-binding* (not chemical)
  neighbourhood add anything the boost misses? In **inductive** expect `meta ≈ raw` by construction
  (cold molecules have no co-binding signature; `zero_signature_mols` quantifies the collapse).
- This is the y-native twin of profile-CF (3b): if it also ties raw, co-binding structure is redundant
  with what raw [mol‖prot] already yields for warm molecules.
- Compare on **AUROC** (prevalence-independent). Whiskers = 95% t-CI over 5 repeats.

Knobs: `K` (co-binding neighbours), `MAX_PARALLEL`. Delete `results/graph/metapath_cf/tables/` to regen.